# BSAN 391 - Two-product Transportation Problem - Gurobi
Source: problem originally developed by Dr. Scott Stevens

In this problem, there is a set $L$ of six locations labelled A through F. Each location is connected to each other one by a shipping link $(i,j)$ forming a set of arcs $N$, and the cost of shipping ($C_{ij}$) one ton of chemical from a location $i$ to another location $j$ is known. For example, it costs 7 dollars to send one ton of chemical from location A to location B, but only 4 dollars to ship a ton from location B to location A.  There are two chemicals, Chemical 1 and Chemical 2, but the shipping cost is the same for each. However, each link has a limited capacity $K_{ij}$.  For example, no more than 9 tons may be shipped from A to B, while no more than 16 tons may be shipped from B to A.

Each location has its own supply of each chemical ($S1_{i},S2_{i}$) and its own demand for each chemical ($D1_{i},D2_{i}$). We need to find the shipping schedule that minimizes overall shipping cost. Let $x_{ij}$ and $y_{ij}$ represent the tons of chemical 1 and chemical 2, respectively, shipped from location $i$ to location $j$.

\begin{equation*}
\begin{aligned}
& {\text{minimize}}
& & \sum_{(i,j) \in N}(x_{ij}+y_{ij})C_{ij} & \color{blue}{\longrightarrow \textbf{Shipping cost minimization (1)}}\\[2mm]
& \text{subject to}&&\\[2mm]
& & & x_{ij}+y_{ij}  \leq  K_{ij}, \text{ for } (i,j) \in N & \color{blue}{\longrightarrow \textbf{Arcs capacity (2)}}\\[2mm]
& & & \sum_{j \in L | i\neq j}x_{ji}-\sum_{j \in L | i\neq j}x_{ij} \geq (D1_{i}-S1_{i}), \text{ for } i \in L & \color{blue}{\longrightarrow \textbf{Chem1 demand satisfaction (3)}}\\[2mm]
& & & \sum_{j \in L | i\neq j}y_{ji}-\sum_{j \in L | i\neq j}y_{ij} \geq (D2_{i}-S2_{i}), \text{ for } i \in L & \color{blue}{\longrightarrow \textbf{Chem2 demand satisfaction (4)}}\\[2mm]
& & & x_{ij},y_{ij}  \in \mathbb{Z}_{0}^{+}, \text{ for } (i,j) \in N & \color{blue}{\longrightarrow \textbf{Nonegative integer variables (5)}}\\[2mm]
\end{aligned}
\end{equation*}

In [1]:
from gurobipy import *

In [2]:
location, demandChem1, supplyChem1, demandChem2, supplyChem2 = multidict( {
    'A': [16, 28, 0, 0],
    'B': [11, 0, 18, 30],
    'C': [0, 22, 12, 15],
    'D': [0, 2, 0, 0],
    'E': [0, 30, 12, 0],
    'F': [6, 0, 0, 0]} )

N = [(i,j) for i in location for j in location if i!=j]    

C = {('A', 'B'):7, ('A', 'C'):9,  ('A', 'D'):6,  ('A', 'E'):5, ('A', 'F'):9,
     ('B', 'A'):4, ('B', 'C'):10, ('B', 'D'):1,  ('B', 'E'):1, ('B', 'F'):4,
     ('C', 'A'):5, ('C', 'B'):8,  ('C', 'D'):1,  ('C', 'E'):1, ('C', 'F'):1,
     ('D', 'A'):6, ('D', 'B'):6,  ('D', 'C'):1,  ('D', 'E'):6, ('D', 'F'):5,
     ('E', 'A'):3, ('E', 'B'):7,  ('E', 'C'):10, ('E', 'D'):6, ('E', 'F'):8,
     ('F', 'A'):8, ('F', 'B'):3,  ('F', 'C'):5,  ('F', 'D'):3, ('F', 'E'):9}

K = {('A', 'B'):9,  ('A', 'C'):17, ('A', 'D'):16, ('A', 'E'):9,  ('A', 'F'):13,
     ('B', 'A'):16, ('B', 'C'):20, ('B', 'D'):16, ('B', 'E'):10, ('B', 'F'):5,
     ('C', 'A'):13, ('C', 'B'):5,  ('C', 'D'):11, ('C', 'E'):12, ('C', 'F'):19,
     ('D', 'A'):19, ('D', 'B'):10, ('D', 'C'):8,  ('D', 'E'):1,  ('D', 'F'):6,
     ('E', 'A'):2,  ('E', 'B'):4,  ('E', 'C'):14, ('E', 'D'):14, ('E', 'F'):6,
     ('F', 'A'):9,  ('F', 'B'):14, ('F', 'C'):16, ('F', 'D'):7,  ('F', 'E'):17}

In [3]:
TwoProduct = Model("TwoProduct")

Set parameter Username
Academic license - for non-commercial use only - expires 2023-08-13


In [4]:
chem1 = TwoProduct.addVars(N, name="chem1flow")
chem2 = TwoProduct.addVars(N, name="chem2flow")

In [5]:
TotalCost = TwoProduct.setObjective(quicksum((chem1[i,j]+chem2[i,j])*C[i,j] for (i,j) in N))

In [6]:
arcCapacity = TwoProduct.addConstrs(chem1[i,j]+chem2[i,j] <= K[i,j] for (i,j) in N)
flowChem1 = TwoProduct.addConstrs(quicksum(chem1[j,i] for j in location if j != i) - quicksum(chem1[i,j] for j in location if j != i) >= (demandChem1[i]-supplyChem1[i]) for i in location)
flowChem2 = TwoProduct.addConstrs(quicksum(chem2[j,i] for j in location if j != i) - quicksum(chem2[i,j] for j in location if j != i) >= (demandChem2[i]-supplyChem2[i]) for i in location)

In [7]:
TwoProduct.optimize()

Gurobi Optimizer version 9.5.2 build v9.5.2rc0 (win64)
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 42 rows, 60 columns and 180 nonzeros
Model fingerprint: 0xfff0712f
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 3e+01]
Presolve time: 0.02s
Presolved: 42 rows, 60 columns, 180 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   2.900000e+01   0.000000e+00      0s
       4    6.2000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.05 seconds (0.00 work units)
Optimal objective  6.200000000e+01


In [8]:
# Display optimal production plan
for v in TwoProduct.getVars():
    if v.x>0:
        print(v.varName, v.x)
    
print("Optimal total shipping cost:", "$"+str(TwoProduct.objVal))

chem1flow[C,F] 17.0
chem1flow[F,B] 11.0
chem2flow[B,E] 10.0
chem2flow[C,E] 2.0
Optimal total shipping cost: $62.0
